# v6 FIX run (GPU 0): continual + gentle per-step noise, from phase 60
Continual run `v4b_continual_s0` continues from phase 60 (Hindi, Chinese, Arabic, ...). After every optimizer step, noise of 0.0007 x each weight matrix's starting std is added (the 'perturb' half of Shrink-and-Perturb; weight decay is the 'shrink'). Question: does the model keep learning after phase 60 instead of breaking down?

**Just Run All.** Output appears below, as in v5. Rerunning resumes from the last saved phase.

In [ ]:
import os, sys, glob, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # run this cell FIRST in a fresh kernel
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
os.system("pkill -f 'train_v6.py.*--gpu 0'")  # stop any background job from the earlier notebooks on this GPU
from common_v6 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
RUN = "v6_cont_noise0.0007_from60"

In [ ]:
# get what this run needs (skips what is already here)
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):                          # retry: HF sometimes drops the connection
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
# 1) the checkpoint we branch from
SRC_RUN, SRC_PHASE = "v4b_continual_s0", 60
if not glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt"):
    name = next(f for f in api.list_repo_files(HF_REPO) if f.startswith(f"{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_"))
    hf_hub_download(HF_REPO, name, local_dir="runs")
INIT = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")[0]
print("branch from:", INIT)
# 2) a leftover folder from the earlier (crashed) attempt with nothing trained yet -> start clean
done = [p for p in glob.glob(f"runs/{RUN}/ckpt/p*.pt") if int(os.path.basename(p)[1:4]) > SRC_PHASE]
if os.path.isdir(f"runs/{RUN}") and not done:
    import shutil; shutil.rmtree(f"runs/{RUN}"); print("removed empty leftover folder")
# 3) FLORES dev + devtest
from datasets import load_dataset
for l, code in {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            os.makedirs("data/flores", exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
# 4) training data must be here
need = [f"data/en-{l}/train.jsonl" for l in ["hi", "zh", "ar"]] + [f"data/en-{l}/val.jsonl" for l in LANGS]
missing = [p for p in need if not os.path.exists(p)]
print("data OK" if not missing else f"MISSING: {missing}")

In [ ]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "zh", "ar"), init_ckpt=INIT, start_phase=SRC_PHASE,
              n_phases=90, seed=0, time_cap_h=120.0, hf_repo=HF_REPO, noise_per_step=7e-4)
print(C)

In [ ]:
D = load_data(train_langs=["hi", "zh", "ar"])

In [ ]:
model = train_run(C, D)

In [ ]:
print_run_summary(RUN)